In [1]:
# 0. Environment Setup

# # Clone the new kauldron repository
# !git clone https://github.com/google-research/kauldron || true
# !pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# # Clone the dialog repository to fix Gemma format issues
# !git clone https://github.com/google-deepmind/dialog || true
# !pip install -q ./dialog

# !pip install -q flax optax seqio



# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'kauldron'...
remote: Enumerating objects: 9553, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 9553 (delta 52), reused 42 (delta 30), pack-reused 9384 (from 3)
Receiving objects: 100% (9553/9553), 2.84 MiB | 33.00 MiB/s, done.
Resolving deltas: 100% (6860/6860), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 58.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB

In [2]:
# %rm -rf /content/Titans_jax

In [3]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 284 (delta 37), reused 67 (delta 36), pack-reused 214 (from 1)
Receiving objects: 100% (284/284), 24.08 MiB | 28.48 MiB/s, done.
Resolving deltas: 100% (167/167), done.
/content/Titans_jax


In [3]:
# import jax
# import jax.numpy as jnp
# import optax
# from kauldron import kd
# import numpy as np
# import os
# import orbax.checkpoint as ocp

# Gemma imports
from gemma import gm
%cd Titans_jax
# Our custom Titans integration
import importlib
import jax
import os
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


[Errno 2] No such file or directory: 'Titans_jax'
/content/Titans_jax
JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


## 3. Load Trained Weights
We didn't save the entire 1B model, just our new memory projections.

In [6]:
import google.colab
import shutil

shutil.unpack_archive('/content/Titans_jax/saved_titans_delta.zip',"./saved_titans_delta")

In [6]:
import orbax.checkpoint as ocp
def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

# Usage:

state = load_titans_weights("./saved_titans_delta")

In [8]:
!ls ./saved_titans_delta


array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [8]:
import jax.numpy as jnp

gemma_config = gm.nn.Gemma3_1B.config

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
model.titans_layer_indices = [11] # Add Titans memory only to the middle layer

# Load original Gemma 3 1B IT weights
print("Loading original Gemma weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

# Merge loaded Titans weights into original Gemma params
print("Merging Titans weights...")
import titans_tree_utils
loaded_titans_params = load_titans_weights("./saved_titans_delta")
merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

tokenizer = gm.text.Gemma3Tokenizer()


Loading original Gemma weights...


Merging Titans weights...


## 4. Dialogue Inference (Test-Time Dynamic Memory)
During inference, the model passes a `cache` containing both the standard Attention KV-cache AND the Titans Neural Memory State. The model automatically updates its internal state.

In [10]:
# We use Gemma's built-in Sampler to correctly handle positions, attention masks, and cache updates
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
output = sampler.sample(prompt, max_new_tokens=20)

print("\n--- Generated Response ---")
print(output)
print("\nMemory state updated automatically in cache.")

Simulation: User enters a turn...

--- Generated Response ---
 společnosti con frecuencia naar des desembark last. Per last entidade. дисци.con do másвремя con

Memory state updated automatically in cache.


In [11]:
sampler.sample("question: Tell me about Colab. Answer:", max_new_tokens=40)

'ale olanixonarınfra ér geçen kolděpodobślin nachSeries (dir content.weekendratriesword फिर后来after collo assim  இன்னtabtrack after a yung nach بعدیafter nachdiv nach product'

In [ ]:
import os
os._exit(0)